[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/toche7/SlideAdvanceDSBDI/blob/main/notebook-12-13-chatbot-groq.ipynb)

**Open this notebook in Google Colab**

# Notebook Demo: Chatbot with GROQ
**BDAT501 Data Science Fundamentals - Module 12-13 (Afternoon)**

Notebook นี้สำหรับสร้าง chatbot ที่คุยได้จริงผ่าน GROQ API และรันได้บน Colab ทันที

In [1]:
%pip install -q groq gradio
print('Install complete')

Install complete


In [2]:
import os
from getpass import getpass
from groq import Groq
import gradio as gr

MODEL_NAME = 'openai/gpt-oss-20b'
SYSTEM_PROMPT = '''You are a helpful Thai-English teaching assistant for a data science class.
Answer clearly, concisely, and politely.
If you are not sure, say what additional data is needed.
'''

print(f'Using model: {MODEL_NAME}')

Using model: openai/gpt-oss-20b


## GROQ API Key Setup (Colab)
1. ไปที่แถบ **Secrets** ใน Colab (ไอคอนกุญแจ)
2. เพิ่ม secret ชื่อ `GROQ_API_KEY`
3. รัน cell ถัดไป

In [3]:
GROQ_API_KEY = None

try:
    from google.colab import userdata
    GROQ_API_KEY = userdata.get('GROQ_API_KEY')
except Exception:
    GROQ_API_KEY = None

if not GROQ_API_KEY:
    GROQ_API_KEY = os.getenv('GROQ_API_KEY', '').strip()

if not GROQ_API_KEY:
    GROQ_API_KEY = getpass('Enter GROQ API key: ').strip()

client = Groq(api_key=GROQ_API_KEY)

test = client.chat.completions.create(
    model=MODEL_NAME,
    messages=[{'role': 'user', 'content': 'Reply with: GROQ ready'}],
    max_tokens=30
)
print('Connection OK:', test.choices[0].message.content)

Connection OK: 


In [4]:
def chat_with_groq(user_message, history):
    messages = [{'role': 'system', 'content': SYSTEM_PROMPT}]

    for turn in history:
        user_text = turn.get('user', '')
        assistant_text = turn.get('assistant', '')
        if user_text:
            messages.append({'role': 'user', 'content': user_text})
        if assistant_text:
            messages.append({'role': 'assistant', 'content': assistant_text})

    messages.append({'role': 'user', 'content': user_message})

    response = client.chat.completions.create(
        model=MODEL_NAME,
        messages=messages,
        temperature=0.4,
        max_tokens=512
    )

    return response.choices[0].message.content

In [5]:
demo_reply = chat_with_groq('อธิบาย Precision กับ Recall แบบสั้นๆ', history=[])
print(demo_reply)

**Precision** (ความแม่นยำ)  
- คือสัดส่วนของผลลัพธ์ที่ถูกทำนายว่าเป็น “บวก” ที่จริง ๆ แล้วเป็นบวก  
- สูตร: \(\displaystyle \text{Precision} = \frac{TP}{TP+FP}\)  
- ค่ามากหมายถึงโมเดลไม่ให้ผลลัพธ์บวกผิดมาก

**Recall** (ความครอบคลุม)  
- คือสัดส่วนของผลลัพธ์บวกจริงทั้งหมดที่โมเดลจับได้  
- สูตร: \(\displaystyle \text{Recall} = \frac{TP}{TP+FN}\)  
- ค่ามากหมายถึงโมเดลจับบวกจริงได้มากที่สุด

โดยทั่วไป Precision และ Recall มี trade‑off: เพิ่ม Precision อาจลด Recall และในทางกลับกัน.


In [6]:
def gradio_chat(message, history):
    normalized_history = []
    for item in history:
        if isinstance(item, dict):
            normalized_history.append({
                'user': item.get('user', item.get('role', '')),
                'assistant': item.get('assistant', item.get('content', ''))
            })
        elif isinstance(item, (list, tuple)) and len(item) == 2:
            normalized_history.append({'user': item[0], 'assistant': item[1]})

    return chat_with_groq(message, normalized_history)

demo = gr.ChatInterface(
    fn=gradio_chat,
    title='BDAT501 Chatbot (GROQ)',
    description='ถามเรื่อง Data Science ได้ทั้งไทยและอังกฤษ',
    examples=[
        'อธิบาย overfitting คืออะไร',
        'ช่วยสรุปขั้นตอนทำ classification project',
        'What is the difference between ROC-AUC and F1?'
    ]
)

demo.launch(share=True)

/usr/local/lib/python3.12/dist-packages/gradio/chat_interface.py:347: UserWarning: The 'tuples' format for chatbot messages is deprecated and will be removed in a future version of Gradio. Please set type='messages' instead, which uses openai-style 'role' and 'content' keys.
  self.chatbot = Chatbot(


Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://b57b1271b8a387fb0a.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
